In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_VGG
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [4]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_VGG()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_9").output,
    outputs=model.output
)

In [8]:
l_label = [2,6,10,13,17,20,24,27,32,35]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_2/VGG-11/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_2/VGG-11/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_2/VGG-11/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_2/VGG-11/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_8: outputs (20000, 256), labels (20000,)
[Saved] re_lu_9: outputs (20000, 256), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data_2/VGG-11/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_5: ou

In [11]:
CACHE_DIR = "./VGG-11/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_2/VGG-11/layer_cache/base")


==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 17454
xi>=0 count: 15353
xi>=0 count: 13874
xi>=0 count: 13057
xi>=0 count: 13039
xi>=0 count: 11205
xi>=0 count: 11761
xi>=0 count: 13695
xi>=0 count: 16409

==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 18392
xi>=0 count: 16934
xi>=0 count: 15040
xi>=0 count: 14149
xi>=0 count: 14353
xi>=0 count: 13680
xi>=0 count: 12637
xi>=0 count: 13796
xi>=0 count: 16333

==== split 0, thr

In [13]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_2/VGG-11/layer_cache/base")

In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_2/VGG-11/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.89296848 0.71061417 0.6250473  0.58782102 0.62135256 0.51020966
 0.5840847  0.68181444 0.86321065]
Layer 1
accuracy: [0.93419778 0.79658272 0.67090422 0.6127008  0.66309484 0.63520537
 0.65236589 0.73703528 0.8685455 ]
Layer 2
accuracy: [0.94342561 0.80582851 0.69062159 0.63903369 0.69725994 0.64306505
 0.62918481 0.73831915 0.84138451]
Layer 3
accuracy: [0.9519313  0.82260106 0.69953626 0.624667   0.70637743 0.6604321
 0.6633743  0.75150733 0.84853527]
Layer 4
accuracy: [0.97727376 0.87661517 0.73839359 0.67991478 0.73540153 0.72301841
 0.72698588 0.79642865 0.83953666]
Layer 5
accuracy: [0.97351064 0.86854021 0.7340802  0.71377103 0.76235795 0.72266046
 0.73055481 0.80718498 0.82506664]
Layer 6
accuracy: [0.9761458  0.95363095 0.84756942 0.82086267 0.8298057  0.8135667
 0.80101274 0.85569485 0.93048416]
Layer 7
accuracy: [0.97150097 0.95637147 0.85592622 0.82107235 0.85637174 0.86396808
 0.8723177  0.89561221 0.93229484]
Layer 8
accuracy: [0.98610802 0.98942618 0

In [16]:
compute_stats(base)

(array([[0.74287665, 0.57312775, 0.70970326],
        [0.80056157, 0.63700034, 0.75264889],
        [0.81329191, 0.65978623, 0.73629616],
        [0.82468954, 0.66382551, 0.7544723 ],
        [0.86409417, 0.71277824, 0.7876504 ],
        [0.85871035, 0.73292981, 0.78760214],
        [0.92578206, 0.82141169, 0.86239725],
        [0.92793289, 0.84713739, 0.90007492],
        [0.97322395, 0.84461391, 0.89553601],
        [0.9793155 , 0.90361511, 0.96117503],
        [1.        , 0.9999221 , 1.        ]]),
 array([[1.11732093e-01, 4.65482230e-02, 1.15646485e-01],
        [1.07525960e-01, 2.06123947e-02, 8.89428462e-02],
        [1.03341651e-01, 2.65489776e-02, 8.66419726e-02],
        [1.03050424e-01, 3.34443331e-02, 7.56207158e-02],
        [9.79234923e-02, 2.37815163e-02, 4.63660243e-02],
        [9.79938906e-02, 2.11229230e-02, 4.09937992e-02],
        [5.60633086e-02, 6.64089975e-03, 5.30685321e-02],
        [5.12896730e-02, 1.86898512e-02, 2.46880724e-02],
        [2.06116852e-02, 4.0

In [17]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/VGG-11/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.7352518  0.56832347 0.48729106 0.4910291  0.58705291 0.41619094
 0.7154074  0.83066795 0.92553185]
Layer 1
accuracy: [0.99889375 0.8266652  0.75011125 0.67792708 0.62118435 0.60583752
 0.71672851 0.83066795 0.07508332]
Layer 2
accuracy: [0.97570981 0.8272194  0.70769815 0.6332762  0.67903672 0.62375717
 0.61146305 0.70990213 0.83163371]
Layer 3
accuracy: [0.99619444 0.83412591 0.74208737 0.64490627 0.68948457 0.67077792
 0.72553996 0.82830796 0.9224042 ]
Layer 4
accuracy: [0.99112345 0.85954738 0.72975678 0.65145644 0.69803366 0.69391472
 0.72138461 0.80822951 0.87448056]
Layer 5
accuracy: [0.99288545 0.86057433 0.73927987 0.68657705 0.70501869 0.69083619
 0.73112438 0.81972004 0.89374587]
Layer 6
accuracy: [0.99428653 0.90162654 0.77303403 0.72955954 0.72720699 0.70039061
 0.69834489 0.80493083 0.92011175]
Layer 7
accuracy: [0.99446882 0.8998297  0.77981347 0.73015021 0.74605019 0.73424563
 0.77725673 0.84963566 0.92999664]
Layer 8
accuracy: [0.99603525 0.93851141

In [18]:
compute_stats(gauss)

(array([[0.59689374, 0.49840359, 0.82377351],
        [0.85855673, 0.63439198, 0.54085681],
        [0.83607372, 0.64477258, 0.71940912],
        [0.85754736, 0.66830151, 0.82344738],
        [0.86157588, 0.67809327, 0.80204148],
        [0.86399007, 0.6963182 , 0.81453389],
        [0.88963816, 0.71803942, 0.806933  ],
        [0.89075482, 0.73670015, 0.85147205],
        [0.92798661, 0.7139796 , 0.82563918],
        [0.92916628, 0.7620112 , 0.86361236],
        [0.93607738, 0.79486321, 0.88563353]]),
 array([[0.10340562, 0.0690245 , 0.08602852],
        [0.10403841, 0.03099957, 0.3325567 ],
        [0.10955439, 0.02483434, 0.09569913],
        [0.10542086, 0.0163011 , 0.08188347],
        [0.10609565, 0.01459301, 0.06575081],
        [0.1031386 , 0.00747239, 0.06474144],
        [0.08842987, 0.02312869, 0.08963459],
        [0.0856801 , 0.00958733, 0.06282456],
        [0.05845421, 0.05897978, 0.07586496],
        [0.05659772, 0.02358523, 0.06439661],
        [0.05091255, 0.02095655,

In [19]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/VGG-11/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.72782845 0.58925228 0.50582742 0.50948651 0.60033948 0.42544552
 0.70098425 0.82230073 0.92490489]
Layer 1
accuracy: [0.99889375 0.8266652  0.749475   0.67116305 0.6372888  0.60582168
 0.71672851 0.83066795 0.11297699]
Layer 2
accuracy: [0.96293237 0.81525411 0.69600669 0.63739724 0.68456346 0.62283815
 0.59019385 0.69971081 0.80509739]
Layer 3
accuracy: [0.98491585 0.83418722 0.72731567 0.64379669 0.69342131 0.67010644
 0.70594259 0.8105324  0.89990693]
Layer 4
accuracy: [0.98382534 0.86524919 0.73123072 0.65773973 0.70966161 0.70591236
 0.72030625 0.80763734 0.8555444 ]
Layer 5
accuracy: [0.98407418 0.86683932 0.74685467 0.69739539 0.72842015 0.71043717
 0.72653192 0.81455668 0.85888157]
Layer 6
accuracy: [0.98587975 0.92843891 0.80607319 0.7717534  0.77082986 0.74495393
 0.75745536 0.82589244 0.92579242]
Layer 7
accuracy: [0.98545344 0.92619886 0.82102345 0.76953497 0.79017251 0.79797099
 0.81814741 0.87009424 0.92479497]
Layer 8
accuracy: [0.99271285 0.96813721

In [20]:
compute_stats(salt)

(array([[0.60808751, 0.51059151, 0.81680176],
        [0.85848304, 0.63862156, 0.55345555],
        [0.82406755, 0.64771453, 0.6958325 ],
        [0.85073574, 0.66841436, 0.8038382 ],
        [0.86116657, 0.69124032, 0.79247111],
        [0.86424325, 0.71356204, 0.80002515],
        [0.90821713, 0.76207523, 0.83489422],
        [0.90964879, 0.78313385, 0.87239224],
        [0.95194941, 0.76801192, 0.85943664],
        [0.95580531, 0.82056133, 0.90690049],
        [0.96604821, 0.86762222, 0.92678017]]),
 array([[0.09267735, 0.07137754, 0.09115937],
        [0.10414548, 0.02723789, 0.31493634],
        [0.10693048, 0.02828259, 0.08912017],
        [0.10453714, 0.02361447, 0.08066672],
        [0.10271414, 0.02217982, 0.0546444 ],
        [0.09843893, 0.01317957, 0.05373697],
        [0.07374347, 0.01280124, 0.07026663],
        [0.06976907, 0.01107341, 0.04542999],
        [0.04079398, 0.05351799, 0.05178536],
        [0.03656171, 0.02357381, 0.04526645],
        [0.02719544, 0.01435074,

In [21]:
SAVE_FILE='e-VGG-11.pkl'

In [22]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [2]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [3]:
SAVE_FILE='e-VGG-11.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [4]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [5]:
mean_var

{'base': {'mean': array([0.67523589, 0.73007027, 0.7364581 , 0.74766245, 0.78817427,
         0.79308077, 0.86986367, 0.89171506, 0.90445795, 0.94803521,
         0.99997403]),
  'std': array([1.21400510e-01, 1.06518172e-01, 1.01115795e-01, 1.00771334e-01,
         8.89813596e-02, 8.10038415e-02, 6.20044977e-02, 4.81602500e-02,
         6.04547359e-02, 4.01153520e-02, 7.34446554e-05]),
  'min': array([0.51020966, 0.6127008 , 0.62918481, 0.624667  , 0.67991478,
         0.71377103, 0.80101274, 0.82107235, 0.78929082, 0.85484228,
         0.9997663 ]),
  'max': array([0.89296848, 0.93419778, 0.94342561, 0.9519313 , 0.97727376,
         0.97351064, 0.9761458 , 0.97150097, 0.98942618, 0.99169447,
         1.        ])},
 'gauss': {'mean': array([0.63969028, 0.67793517, 0.73341848, 0.78309875, 0.78057021,
         0.79161405, 0.8048702 , 0.82630901, 0.82253513, 0.85159662,
         0.87219138]),
  'std': array([0.16180041, 0.24199832, 0.11600242, 0.11318598, 0.10538322,
         0.0995486 ,